# 06 — Chất lượng câu hỏi: đo trước, sửa sau

Câu hỏi thật sự cần trả lời: *một bộ câu hỏi sinh ra có dùng được không*, và
*đổi model có làm nó khá hơn không*. Trước đây tôi đo bằng phép kiểm ngây thơ —
"đáp án có nằm nguyên văn trong đoạn nguồn không" — và nó chấm sai cả hai chiều:

- Câu hỏi phủ định ("điều nào **KHÔNG** được nêu trong tài liệu") có đáp án cố ý
  *không* nằm trong nguồn. Phép kiểm cũ cho nó 0 điểm dù câu hỏi hoàn toàn đúng.
- Một câu hỏi hỏng vì **phương án nhiễu cũng đúng** lại được cho là tốt, chỉ vì
  đáp án của nó chép lại từ nguồn.

`ml_pipeline/question_quality.py` đo cái đáng đo: nguồn có khẳng định đáp án
không, có nhiễu nào cũng đúng theo nguồn không, nhiễu có nhại lại đáp án không —
và lật ngược phép kiểm cho câu phủ định. Không dùng model ngôn ngữ (TF-IDF trên
n-gram ký tự + từ), nên chạy được cho cả tiếng Việt và cho cùng con số mọi lần.

Dữ liệu vào là file dump từ nền tảng (nội dung tài liệu của người dùng nên không
commit) — sinh bằng:

```
cd backend && ./venv/bin/python manage.py dump_question_sets \
    --excerpts 3 --count 5 --providers ollama gemini
```

Command đó dùng **đúng prompt production** (`notebooks.services.generate.quiz_prompt`)
và không ghi gì vào database, nên đo không làm bẩn quiz của người học.

In [1]:
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
from ml_pipeline.notebook_setup import bootstrap
bootstrap()
from ml_pipeline.question_quality import score_set, SUPPORTED, RIVAL, ECHO

ROOT = Path.cwd().resolve().parents[1]
dump = json.load(open(ROOT / 'ml_pipeline' / 'outputs' / 'question_sets.json'))

print(f"ngưỡng: nguồn-khẳng-định {SUPPORTED} | nhiễu-cũng-đúng {RIVAL} | nhiễu-nhại-đáp-án {ECHO}")
print(f"{len(dump['stored'])} câu đã lưu có trích đoạn nguồn")
for name, sets in dump['generated'].items():
    print(f"  {name}: {len(sets)} bộ, {sum(len(s['items']) for s in sets)} câu mới sinh")

ngưỡng: nguồn-khẳng-định 0.45 | nhiễu-cũng-đúng 0.45 | nhiễu-nhại-đáp-án 0.72
7 câu đã lưu có trích đoạn nguồn
  gemini:gemini-2.5-flash: 3 bộ, 15 câu mới sinh
  gemini:gemini-3.5-flash: 0 bộ, 0 câu mới sinh
  gemini:gemini-3.8-flash: 2 bộ, 10 câu mới sinh
  gemini:gemini-3.1-flash-lite: 3 bộ, 15 câu mới sinh


## Hiện trạng: những câu người học đã thật sự nhận được

Đây là điểm khởi đầu — câu hỏi do model cũ (`llama3.2:3b`) sinh ra và đã nằm
trong quiz. Chỉ những câu có trích đoạn nguồn kèm theo mới chấm được (provenance
thêm vào sau, nên số này nhỏ).

In [2]:
import pandas as pd

def scorecard(groups: dict[str, list[tuple[list[dict], str]]]) -> pd.DataFrame:
    """Một dòng mỗi nhóm: tỉ lệ đạt và những lỗi hay gặp."""
    rows = []
    for name, batches in groups.items():
        agg = {'total': 0, 'scored': 0, 'duplicates': 0, 'passed': 0}
        flags: dict[str, int] = {}
        for items, excerpt in batches:
            card = score_set(items, excerpt)
            for key in agg:
                agg[key] += card[key]
            for flag, n in card['flag_counts'].items():
                flags[flag] = flags.get(flag, 0) + n
        rows.append({
            'bộ': name,
            'số câu': agg['total'],
            'đạt': agg['passed'],
            'tỉ lệ đạt': round(agg['passed'] / agg['total'], 3) if agg['total'] else 0.0,
            'trùng lặp': agg['duplicates'],
            **{f'lỗi: {f}': n for f, n in sorted(flags.items(), key=lambda kv: -kv[1])},
        })
    return pd.DataFrame(rows).fillna(0)

stored_batches = [([r], r['excerpt']) for r in dump['stored']]
scorecard({'đã lưu (llama3.2:3b)': stored_batches})

,bộ,số câu,đạt,tỉ lệ đạt,trùng lặp,lỗi: answer_unsupported,lỗi: rival_distractor,lỗi: negative_answer_supported,lỗi: negative_distractors_unsupported
0,đã lưu (llama3.2:3b),7,1,0.143,0,3,2,1,1


## Đối đầu: cùng đoạn nguồn, cùng prompt, khác model

Mỗi model nhận **cùng những đoạn nguồn** và cùng prompt, nên chênh lệch là của
model chứ không của tài liệu. Câu trùng nhau trong một bộ tính là trượt: bộ đề
có hai câu giống nhau là bộ đề kém.

In [3]:
groups = {'đã lưu (llama3.2:3b)': stored_batches}
for name, sets in dump['generated'].items():
    groups[f'sinh mới ({name})'] = [(s['items'], s['excerpt']) for s in sets]

board = scorecard(groups)
board

,bộ,số câu,đạt,tỉ lệ đạt,trùng lặp,lỗi: answer_unsupported,lỗi: rival_distractor,lỗi: negative_answer_supported,lỗi: negative_distractors_unsupported
0,đã lưu (llama3.2:3b),7,1,0.143,0,3.0,2.0,1.0,1.0
1,sinh mới (gemini:gemini-2.5-flash),15,14,0.933,0,0.0,1.0,0.0,0.0
2,sinh mới (gemini:gemini-3.5-flash),0,0,0.000,0,0.0,0.0,0.0,0.0
3,sinh mới (gemini:gemini-3.8-flash),10,10,1.000,0,0.0,0.0,0.0,0.0
4,sinh mới (gemini:gemini-3.1-flash-lite),15,13,0.867,0,1.0,1.0,0.0,0.0


In [4]:
# Sinh nhanh hay chậm cũng là một phần của "dùng được không".
for name, sets in dump['generated'].items():
    if sets:
        secs = [s['seconds'] for s in sets]
        print(f"{name}: {sum(secs):.0f}s cho {len(sets)} bộ "
              f"({sum(secs) / len(secs):.0f}s mỗi bộ)")

gemini:gemini-2.5-flash: 32s cho 3 bộ (11s mỗi bộ)
gemini:gemini-3.8-flash: 16s cho 2 bộ (8s mỗi bộ)
gemini:gemini-3.1-flash-lite: 17s cho 3 bộ (6s mỗi bộ)


## Lỗi nào chiếm đa số — đó mới là việc cần sửa

Nếu cả hai model cùng trượt vì một lỗi, thì đổi model không chữa được lỗi đó;
phải sửa ở chỗ khác (prompt, hoặc một vòng tự kiểm sau khi sinh rồi loại câu
hỏng). Bảng dưới xếp lỗi theo số lần xuất hiện.

In [5]:
def flag_table(groups: dict) -> pd.DataFrame:
    rows = []
    for name, batches in groups.items():
        for items, excerpt in batches:
            for report in score_set(items, excerpt)['reports']:
                for flag in report.flags:
                    rows.append({'bộ': name, 'lỗi': flag})
    if not rows:
        return pd.DataFrame()
    return (pd.DataFrame(rows).value_counts(['lỗi', 'bộ']).unstack(fill_value=0)
            .assign(tổng=lambda d: d.sum(axis=1)).sort_values('tổng', ascending=False))

flag_table(groups)

bộ,đã lưu (llama3.2:3b),sinh mới (gemini:gemini-2.5-flash),sinh mới (gemini:gemini-3.1-flash-lite),tổng
lỗi,,,,
answer_unsupported,3,0,1,4
rival_distractor,2,1,1,4
negative_answer_supported,1,0,0,1
negative_distractors_unsupported,1,0,0,1


### Đọc vài câu trượt bằng mắt

Con số chỉ đáng tin khi những câu nó gạch đi thật sự là câu tệ. Cell dưới in ra
câu hỏi, đáp án, và mức mà nguồn khẳng định từng phương án — để tự kiểm tra xem
phép chấm có công bằng không.

In [6]:
def show_failures(name: str, limit: int = 4) -> None:
    shown = 0
    for items, excerpt in groups.get(name, []):
        for report in score_set(items, excerpt)['reports']:
            if report.ok or shown >= limit:
                continue
            shown += 1
            print(f"\n[{', '.join(report.flags)}]")
            print(f"  hỏi   : {report.question[:110]}")
            print(f"  đáp án: (nguồn khẳng định {report.answer_support})")
            for d, s in report.distractor_support.items():
                mark = ' ← cũng đúng' if d in report.rivals else (' ← nhại đáp án' if d in report.echoes else '')
                print(f"  nhiễu : {s:<6} {str(d)[:70]}{mark}")

for name in groups:
    print('=' * 70, '\n', name)
    show_failures(name)

 đã lưu (llama3.2:3b)

[answer_unsupported]
  hỏi   : What was the final sample size of national AI policy strategies after screening?
  đáp án: (nguồn khẳng định 0.042)
  nhiễu : 0.0    15
  nhiễu : 0.0    10
  nhiễu : 0.0    30

[negative_answer_supported, negative_distractors_unsupported]
  hỏi   : What is one way that a device's value to a vulnerable user cannot be assessed in isolation from its material c
  đáp án: (nguồn khẳng định 0.65)
  nhiễu : 0.045  by functionality
  nhiễu : 0.007  by cost
  nhiễu : 0.042  by aesthetics

[answer_unsupported]
  hỏi   : How does CityHelp's decision to let users act quickly without being forced through account creation or data en
  đáp án: (nguồn khẳng định 0.223)
  nhiễu : 0.21   by improving overall well-being
  nhiễu : 0.348  by increasing the risk of accidents
  nhiễu : 0.14   by making emergency contact less visible

[answer_unsupported]
  hỏi   : How does the fragility of the mobile context affect phone design?
  đáp án: (nguồn khẳng địn


[rival_distractor]
  hỏi   : What key insight does the article illustrate regarding the assessment of a device's value to a vulnerable user
  đáp án: (nguồn khẳng định 0.833)
  nhiễu : 0.217  A device’s value is solely determined by its technological specificati
  nhiễu : 0.383  The most valuable devices are those with the broadest general appeal.
  nhiễu : 0.589  Design should always assume an ideal, well-resourced context for users ← cũng đúng


 sinh mới (gemini:gemini-3.5-flash)
 sinh mới (gemini:gemini-3.8-flash)


 sinh mới (gemini:gemini-3.1-flash-lite)



[rival_distractor]
  hỏi   : How should design be approached for vulnerable users according to the paper?
  đáp án: (nguồn khẳng định 0.61)
  nhiễu : 0.011  By focusing on aesthetic minimalism
  nhiễu : 0.602  By assuming an ideal and well-resourced context ← cũng đúng
  nhiễu : 0.132  By prioritizing features for wealthy donors



[answer_unsupported]
  hỏi   : What type of experience is mentioned as an example of difficulties with mental and conceptual models?
  đáp án: (nguồn khẳng định 0.414)
  nhiễu : 0.073  Designing a website interface
  nhiễu : 0.307  Writing a report on user performance statistics
  nhiễu : 0.547  Creating a physical prototype of a rain boot


## Lưu bảng điểm

`reports/question_quality.json` là mốc so sánh: mỗi lần đổi model, đổi prompt,
hay thêm vòng tự kiểm, chạy lại notebook này và so với con số cũ.

In [7]:
from datetime import date

report = {
    'measured_on': str(date.today()),
    'thresholds': {'supported': SUPPORTED, 'rival': RIVAL, 'echo': ECHO},
    'questions_per_excerpt': dump['count_requested'],
    'groups': json.loads(board.to_json(orient='records', force_ascii=False)),
}
out = ROOT / 'ml_pipeline' / 'reports' / 'question_quality.json'
out.write_text(json.dumps(report, ensure_ascii=False, indent=1))
print('đã lưu', out)
print(json.dumps(report['groups'], ensure_ascii=False, indent=1)[:900])

đã lưu /Users/lagiavi/School/Year 3/FYP/Study Platform Development/ml_pipeline/reports/question_quality.json
[
 {
  "bộ": "đã lưu (llama3.2:3b)",
  "số câu": 7,
  "đạt": 1,
  "tỉ lệ đạt": 0.143,
  "trùng lặp": 0,
  "lỗi: answer_unsupported": 3.0,
  "lỗi: rival_distractor": 2.0,
  "lỗi: negative_answer_supported": 1.0,
  "lỗi: negative_distractors_unsupported": 1.0
 },
 {
  "bộ": "sinh mới (gemini:gemini-2.5-flash)",
  "số câu": 15,
  "đạt": 14,
  "tỉ lệ đạt": 0.933,
  "trùng lặp": 0,
  "lỗi: answer_unsupported": 0.0,
  "lỗi: rival_distractor": 1.0,
  "lỗi: negative_answer_supported": 0.0,
  "lỗi: negative_distractors_unsupported": 0.0
 },
 {
  "bộ": "sinh mới (gemini:gemini-3.5-flash)",
  "số câu": 0,
  "đạt": 0,
  "tỉ lệ đạt": 0.0,
  "trùng lặp": 0,
  "lỗi: answer_unsupported": 0.0,
  "lỗi: rival_distractor": 0.0,
  "lỗi: negative_answer_supported": 0.0,
  "lỗi: negative_distractors_unsupported": 0.0
 },
 {
  "bộ": "sinh mới (gemini:gemini-3.8-flash)",
  "số câu": 10,
  "đạt": 10,
  "